# Metadata

Domain: Mathematiccs

Subdomain: Convex Optimization / First-Order Methods / Sparse Regression

Paper reference: A. Beck and M. Teboulle, "A Fast Iterative Shrinkage-Thresholding Algorithm for Linear Inverse Problems," SIAM Journal on Imaging Sciences, 2(1), 183-202, 2009. https://epubs.siam.org/doi/10.1137/080716542

Required Dependencies: numpy>=1.21 (used by all reference solutions and tests, e.g. np.linalg.norm, np.sign, np.maximum); math (Python standard library, used by Subproblem 2, the Main Problem, and their tests, e.g. math.sqrt for the FISTA momentum sequence); pytest>=7.0 (test runner, provided by the standard project environment).

# Title

FISTA for L1-Regularized Least Squares: From the Proximal Shrinkage Step to an Accelerated, O(1/k^2)-Convergent Solver

# Subproblem 1

## Prompt

Consider the L1-regularized least squares (LASSO) problem

    min_{x in R^n}  F(x) = 1/2 ||Ax - b||_2^2 + lam * ||x||_1.

The smooth part has gradient grad f(x) = A^T(Ax-b) with Lipschitz
constant L = ||A||_2^2. The non-smooth term lam*||x||_1 is handled by
the proximal operator, which for the L1 norm has the closed form of
component-wise soft-thresholding.

Implement two functions:

(1) prox_gradient_step(x, grad, L, lam, weights=None) — one proximal-
    gradient (ISTA) step for the LASSO objective, possibly with a
    per-coordinate weighting of the L1 term.

(2) prox_l1_weighted(u, theta, weights=None) — the proximal operator of
    the weighted L1 norm,

        prox_{theta * sum_i(w_i * |x_i|)}(u)
            = argmin_x  1/2 * ||x - u||_2^2
                       + theta * sum_i(w_i * |x_i|),

    where theta >= 0 is a scalar and w_i >= 0 are per-coordinate weights.

```python
def prox_gradient_step(x, grad, L, lam, weights=None):
    """
    One proximal-gradient (ISTA) step, with optional per-coordinate
    weighting of the L1 regularizer.

    The step is

        x_next = prox_{(lam/L) * sum_i weights[i] |.|}(x - grad / L)

    i.e. a gradient step of size 1/L on the smooth part, then a
    proximal step with shrinkage coefficient lam/L (not lam). The
    coefficient is lam/L because the proximal subproblem is scaled by
    the same 1/L factor as the gradient step: at convergence the
    fixed point satisfies 0 in grad f(x) + lam * d(||x||_1), and the
    ISTA iteration achieves that only when the L1 term is weighted by
    lam/L inside the prox.

    Parameters
    ----------
    x : np.ndarray, shape (n,)
        Current point at which the gradient step is taken.
    grad : np.ndarray, shape (n,)
        Gradient of the smooth part at x, i.e. grad = A^T(A @ x - b).
        Must have the same shape as x.
    L : float
        Step-size denominator (Lipschitz constant of grad f). Positive
        and finite.
    lam : float
        L1 regularization strength. Non-negative and finite. lam = 0
        is the plain (unshrunk) gradient step.
    weights : np.ndarray, shape (n,), optional
        Non-negative per-coordinate weights. If None, defaults to all
        ones (the uniform L1 regularizer lam*||x||_1). When supplied,
        the regularizer is lam * sum_i(weights[i] * |x_i|).

    Returns
    -------
    x_next : np.ndarray, shape (n,)
        prox_{(lam/L) * sum_i weights[i] |.|}(x - grad / L).

    Raises
    ------
    ValueError
        If x and grad are not finite 1-D arrays of the same shape; if L
        is not positive and finite; if lam is not non-negative and
        finite; if weights is supplied but is not a finite, non-negative
        1-D array of the same shape as x.
    """


def prox_l1_weighted(u, theta, weights=None):
    """
    Proximal operator of the weighted L1 norm.

    Parameters
    ----------
    u : np.ndarray, shape (n,)
        Point at which the proximal operator is evaluated. Finite 1-D
        array.
    theta : float
        Non-negative scalar coefficient. Non-negative and finite. Note
        that theta already incorporates any scaling: the ISTA step of
        Subproblem 1 calls this function with theta = lam / L.
    weights : np.ndarray, shape (n,), optional
        Non-negative per-coordinate weights. If None, defaults to all
        ones (the standard unweighted L1 proximal operator).

    Returns
    -------
    x : np.ndarray, shape (n,)
        The unique minimizer of 1/2 ||x - u||_2^2
        + theta * sum_i(w_i * |x_i|).

    Raises
    ------
    ValueError
        If u is not a finite 1-D array; if theta is not non-negative and
        finite; if weights is supplied but is not a finite, non-negative
        1-D array of the same shape as u.
    """
    ...
    return x

## Background

**Why soft-thresholding, and not just "round small values to zero."** The proximal operator of $\theta\|\cdot\|_1$ is, by definition,

$$
\mathrm{prox}_{\theta\|\cdot\|_1}(u) \;=\; \arg\min_x\ \tfrac12\|x-u\|_2^2 + \theta\|x\|_1.
$$

Solving this 1-D minimization coordinate-wise gives the shrinkage formula $S_\theta(u)_i = \mathrm{sign}(u_i)\max(|u_i|-\theta,0)$. It is not an ad hoc rounding rule — it is the exact solution of the sub-problem that ISTA/FISTA reduce to at every iteration.

**Scaling: the shrinkage coefficient is $\lambda/L$, not $\lambda$.** The proximal-gradient step takes a gradient step of size $1/L$ on the smooth part and then applies the proximal operator of the *scaled* non-smooth term $(1/L)\,\lambda\|\cdot\|_1$:

$$
x_{k+1} \;=\; \mathrm{prox}_{(1/L)\,\lambda\|\cdot\|_1}\!\Bigl(x_k - \tfrac{1}{L}\nabla f(x_k)\Bigr).
$$

Because $\|\cdot\|_1$ is $1$-homogeneous, $\mathrm{prox}_{t\|\cdot\|_1}(v) = S_t(v)$ for any $t \geq 0$, so the inner threshold is $t = \lambda/L$. The shrinkage applied to $x_k - \nabla f(x_k)/L$ therefore uses coefficient $\lambda/L$, not $\lambda$. The two coincide only in the degenerate case $L = 1$.

The distinction is not cosmetic. The fixed point of the iteration satisfies $0 \in \nabla f(x^\star) + \lambda\,\partial\|x^\star\|_1$. At that point, the gradient step gives $z_k = x_k - \nabla f(x_k)/L$, and the shrinkage threshold that leaves $x_k$ unchanged is exactly $\lambda/L$: if the threshold were $\lambda$, the iterates would converge to the minimizer of a differently scaled problem.

## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np


def test_case_1():
    x = np.array([1.0, -2.0, 0.5])
    grad = np.array([0.4, -0.2, 0.1])
    L = 2.0
    x_next = prox_gradient_step(x, grad, L, 0.0)
    assert np.allclose(x_next, x - grad / L, atol=1e-10)
    print('test_case_1 passed')


def test_case_2():
    x = np.array([0.0, 0.0, 0.0])
    grad = np.array([-3.0, 3.0, 0.5])
    L = 1.0
    lam = 1.0
    x_next = prox_gradient_step(x, grad, L, lam)
    assert np.allclose(x_next, [2.0, -2.0, 0.0], atol=1e-10)
    print('test_case_2 passed')


def test_case_3():
    x = np.array([2.0, -1.0, 0.3, -0.3])
    grad = np.array([1.0, -4.0, 0.6, 0.6])
    L = 4.0
    lam = 1.2
    z = x - grad / L
    expected = np.sign(z) * np.maximum(np.abs(z) - lam / L, 0.0)
    x_next = prox_gradient_step(x, grad, L, lam)
    assert np.allclose(x_next, expected, atol=1e-10)
    print('test_case_3 passed')


def test_case_4():
    x = np.array([0.0])
    grad = np.array([-2.0])
    L = 1.0
    lam = 2.0
    x_next = prox_gradient_step(x, grad, L, lam)
    assert np.allclose(x_next, [0.0], atol=1e-12)
    print('test_case_4 passed')


def test_case_5():
    x = np.zeros(9)
    grad = -np.array([-4.0, -3.0, -2.0, -1.0, -0.1, 0.0, 1.0, 2.5, 5.0])
    L = 1.5
    lam = 0.7
    z = x - grad / L
    x_next = prox_gradient_step(x, grad, L, lam)
    assert np.all(np.abs(x_next) <= np.abs(z) + 1e-12)
    nonzero = x_next != 0
    assert np.all(np.sign(x_next[nonzero]) == np.sign(z[nonzero]))
    print('test_case_5 passed')


def test_case_6():
    x = np.array([1.0, 2.0])
    grad = np.array([0.1, 0.2])

    for kwargs, label in [
        (dict(L=0.0, lam=0.5), 'L=0'),
        (dict(L=-2.0, lam=0.5), 'negative L'),
        (dict(L=1.0, lam=-0.1), 'negative lam'),
    ]:
        try:
            prox_gradient_step(x, grad, **kwargs)
            assert False, f'expected ValueError for {label}'
        except ValueError:
            pass

    try:
        prox_gradient_step(np.array([1.0, np.nan]), grad, L=1.0, lam=0.5)
        assert False, 'expected ValueError for non-finite x'
    except ValueError:
        pass

    try:
        prox_gradient_step(x, np.array([0.1, 0.2, 0.3]), L=1.0, lam=0.5)
        assert False, 'expected ValueError for mismatched shapes'
    except ValueError:
        pass

    print('test_case_6 passed')


def test_case_7():
    u = np.array([3.0, -2.0, 0.5, -0.3])
    theta = 0.8
    x = prox_l1_weighted(u, theta)
    expected = np.sign(u) * np.maximum(np.abs(u) - theta, 0.0)
    assert np.allclose(x, expected, atol=1e-12)
    print('test_case_7 passed')


def test_case_8():
    u = np.array([2.0, -3.0, 0.5])
    theta = 1.0
    weights = np.array([0.5, 2.0, 0.0])
    x = prox_l1_weighted(u, theta, weights=weights)
    expected = np.array([
        np.sign(2.0) * max(abs(2.0) - 1.0 * 0.5, 0.0),
        np.sign(-3.0) * max(abs(-3.0) - 1.0 * 2.0, 0.0),
        np.sign(0.5) * max(abs(0.5) - 1.0 * 0.0, 0.0),
    ])
    assert np.allclose(x, expected, atol=1e-12)
    assert np.isclose(x[2], 0.5, atol=1e-15)
    print('test_case_8 passed')


def test_case_9():
    u = np.array([5.0, -5.0, 0.4])
    theta = 0.2
    weights_zero = np.zeros(3)
    x = prox_l1_weighted(u, theta, weights=weights_zero)
    assert np.allclose(x, u, atol=1e-15)
    print('test_case_9 passed')


def test_case_10():
    u = np.array([3.0, -2.0, 0.5, -0.3])
    theta = 0.8
    weights = np.array([1.0, 1.0, 1.0, 1.0])
    x_uniform = prox_l1_weighted(u, theta)
    x_weighted = prox_l1_weighted(u, theta, weights=weights)
    assert np.allclose(x_uniform, x_weighted, atol=1e-15)
    print('test_case_10 passed')


def test_case_11():
    x = np.array([1.0, -2.0, 0.5])
    grad = np.array([0.4, -0.2, 0.1])
    L = 2.0
    lam = 0.6
    weights = np.array([0.5, 1.0, 2.0])
    x_next = prox_gradient_step(x, grad, L, lam, weights=weights)
    z = x - grad / L
    expected = np.sign(z) * np.maximum(np.abs(z) - lam * weights / L, 0.0)
    assert np.allclose(x_next, expected, atol=1e-12)
    print('test_case_11 passed')


def test_case_12():
    u = np.zeros(9)
    u[:] = [-4.0, -3.0, -2.0, -1.0, -0.1, 0.0, 1.0, 2.5, 5.0]
    theta = 0.7
    weights = np.array([0.5, 1.0, 1.5, 2.0, 0.3, 1.0, 0.0, 1.0, 0.5])
    x = prox_l1_weighted(u, theta, weights=weights)
    assert np.all(np.abs(x) <= np.abs(u) + 1e-12)
    nonzero = x != 0
    assert np.all(np.sign(x[nonzero]) == np.sign(u[nonzero]))
    print('test_case_12 passed')


def test_case_13():
    u = np.array([1.0, 2.0])
    try:
        prox_l1_weighted(u, theta=-0.1)
        assert False, 'expected ValueError for negative theta'
    except ValueError:
        pass
    try:
        prox_l1_weighted(u, theta=0.5, weights=np.array([1.0, -0.5]))
        assert False, 'expected ValueError for negative weights'
    except ValueError:
        pass
    try:
        prox_l1_weighted(u, theta=0.5, weights=np.array([1.0, 0.5, 0.3]))
        assert False, 'expected ValueError for mismatched weights shape'
    except ValueError:
        pass
    print('test_case_13 passed')

## Solution

In [ ]:
import numpy as np


def prox_l1_weighted(u, theta, weights=None):
    """Proximal operator of the weighted L1 norm."""
    u = np.asarray(u, dtype=float)
    if u.ndim != 1:
        raise ValueError("u must be a 1-D array")
    if not np.all(np.isfinite(u)):
        raise ValueError("u must contain only finite values")
    if not np.isfinite(theta) or theta < 0:
        raise ValueError(f"theta must be non-negative and finite, got {theta!r}")

    if weights is None:
        w = np.ones_like(u)
    else:
        w = np.asarray(weights, dtype=float)
        if w.ndim != 1 or w.shape != u.shape:
            raise ValueError("weights must be a 1-D array of the same shape as u")
        if not np.all(np.isfinite(w)):
            raise ValueError("weights must contain only finite values")
        if np.any(w < 0):
            raise ValueError("weights must be non-negative")

    return np.sign(u) * np.maximum(np.abs(u) - theta * w, 0.0)


def prox_gradient_step(x, grad, L, lam, weights=None):
    """One proximal-gradient (ISTA) step, with optional weighted L1."""
    x = np.asarray(x, dtype=float)
    grad = np.asarray(grad, dtype=float)

    if x.ndim != 1 or grad.ndim != 1 or x.shape != grad.shape:
        raise ValueError("x and grad must be 1-D arrays of the same shape")
    if not np.all(np.isfinite(x)):
        raise ValueError("x must contain only finite values")
    if not np.all(np.isfinite(grad)):
        raise ValueError("grad must contain only finite values")
    if not np.isfinite(L) or L <= 0:
        raise ValueError(f"L must be a positive finite number, got L={L!r}")
    if not np.isfinite(lam) or lam < 0:
        raise ValueError(f"lam must be a non-negative finite number, got lam={lam!r}")

    z = x - grad / L
    return prox_l1_weighted(z, lam / L, weights=weights)

# Subproblem 2

## Prompt

Plain proximal gradient descent (ISTA) converges at rate O(1/k) in
objective value. Beck and Teboulle's FISTA momentum extrapolation
improves this to O(1/k^2) at no extra per-iteration cost:

    t_{k+1} = (1 + sqrt(1 + 4 t_k^2)) / 2,
    y_{k+1} = x_k + ((t_k - 1) / t_{k+1}) * (x_k - x_{k-1}),

initialized with t_1 = 1 (so y_1 = x_0, no extrapolation on the first
step).

The FISTA acceleration is not monotone in the objective. When it
oscillates, resetting the momentum parameter damps that oscillation and
empirically restores the steady, ISTA-like descent on the next few steps
(the reset is a convergence heuristic, not a guarantee of a monotone
objective). The O'Donoghue-Candes restart rule triggers a reset whenever
the objective strictly increases relative to the previous iterate.

Implement three functions:

(1) fista_momentum_update(t_k, x_prev, x_curr) — the standard FISTA
    momentum update.

(2) fista_should_restart(obj_history) — the O'Donoghue-Candes restart
    check.

(3) fista_step_momentum(t_k, x_prev, x_curr, obj_history) — combined
    momentum update that applies the restart rule from (2) before
    running the update from (1).

    Validation order (explicit). The caller's t_k is validated against
    the same rules used by fista_momentum_update -- numeric scalar,
    finite, >= 1 -- BEFORE the restart override is applied. If
    validation fails, ValueError is raised regardless of whether the
    restart condition would have been triggered. Only after validation
    passes is the caller's t_k possibly overridden by 1 for the update.

    Consequences.
      - When the restart condition is triggered AND the caller's t_k is
        valid, the effective momentum parameter used by the recurrence
        is t_k = 1. The returned t_next equals the golden ratio
        (1 + sqrt(5))/2, the returned y_next equals x_curr exactly, and
        the returned third value is True.
      - When the restart condition is NOT triggered, the function
        returns exactly the same outputs as
        fista_momentum_update(t_k, x_prev, x_curr), and the third value
        is False.
      - When the caller's t_k is invalid (non-numeric, non-finite, or
        < 1), the function raises ValueError regardless of obj_history.

```python
def fista_momentum_update(t_k, x_prev, x_curr):
    """
    Standard FISTA momentum update. Given the current momentum parameter
    t_k and the two most recent proximal-gradient iterates x_{k-1}
    (x_prev) and x_k (x_curr), advance the momentum sequence and form the
    extrapolated point:

        t_next = (1 + sqrt(1 + 4 t_k^2)) / 2,
        y_next = x_curr + ((t_k - 1) / t_next) * (x_curr - x_prev).

    Parameters
    ----------
    t_k : float
        Current momentum parameter. Must be a finite numeric scalar with
        t_k >= 1; otherwise ValueError is raised.
    x_prev, x_curr : np.ndarray, shape (n,)
        The previous and current iterates. Must be 1-D arrays of the same
        shape containing only finite values; otherwise ValueError is
        raised.

    Returns
    -------
    t_next : float
        The next momentum parameter (strictly greater than t_k).
    y_next : np.ndarray, shape (n,)
        The extrapolated point. Equals x_curr exactly when t_k == 1 or
        when x_prev == x_curr (no extrapolation).
    """
    ...
    return t_next, y_next


def fista_should_restart(obj_history):
    """
    O'Donoghue-Candes function-value restart check. Returns True when the
    objective strictly increased on the most recent step, i.e. when the
    last recorded objective value exceeds the one before it, signalling
    that the FISTA momentum should be reset.

    Parameters
    ----------
    obj_history : np.ndarray, shape (m,)
        Non-empty 1-D array of recorded objective values, in iteration
        order, containing only finite values; otherwise ValueError is
        raised.

    Returns
    -------
    restart : bool
        True if obj_history has at least two entries and the last is
        strictly greater than the second-to-last; False otherwise
        (including the single-entry case, where no increase can be
        observed).
    """
    ...
    return restart


def fista_step_momentum(t_k, x_prev, x_curr, obj_history):
    """
    Combined FISTA momentum update with the O'Donoghue-Candes restart
    rule. See the prompt above for the exact validation-order semantics.

    Returns
    -------
    t_next : float
    y_next : np.ndarray, shape (n,)
    restarted : bool
    """
    ...
    return t_next, y_next, restarted

## Background

**A closed-form sanity check hiding in the recurrence.** Starting the sequence at $t_1 = 1$ and applying the recurrence once gives $t_2 = (1 + \sqrt{1+4})/2 = (1 + \sqrt5)/2$ — the golden ratio $\varphi \approx 1.6180339887$.

**Why $t_k \ge 1$ is required.** The rate guarantee $F(x_k)-F^\star \le O(1/k^2)$ in Beck and Teboulle's analysis is derived for the sequence generated from $t_1 = 1$ by the stated recurrence. Feeding the formula a $t_k < 1$ produces a value outside that trajectory; the domain boundary $t_k \ge 1$ is the natural one for the formula's intended use.

**A structural invariant across the whole valid domain.** For every valid $t_k \ge 1$, the recurrence must produce $t_{k+1} > t_k$ — the momentum parameter is strictly increasing along any legitimate FISTA run.

**The degenerate case.** If two consecutive proximal-gradient outputs are identical, the extrapolation term vanishes and $y_{k+1}$ must come back exactly equal to $x_k$ regardless of $t_k$.

**The O'Donoghue–Candès restart rule.** Although FISTA achieves $O(1/k^2)$ globally, the objective sequence $F(x_k)$ is not monotone — it can oscillate. Restarting the momentum whenever $F$ increases damps that oscillation and empirically restores steady, ISTA-like descent on the following steps; it is a practical convergence heuristic and does not by itself make the objective provably monotone (that would require the separate monotone-FISTA construction). The rule is *strict*: restart iff $F(x_k) > F(x_{k-1})$. When $F(x_k) = F(x_{k-1})$, no restart is triggered.

**What "restart" does to the momentum parameter — the subtle part.** There are two natural implementations of a restart, and they disagree on the momentum parameter even though they agree on the extrapolated point.

*Correct (Beck–Teboulle / O'Donoghue–Candès).* The restart overrides the CURRENT momentum parameter $t_k$ with $1$ before the recurrence is applied:

$$
t_{k+1} \;=\; \frac{1 + \sqrt{1 + 4\cdot 1^2}}{2} \;=\; \frac{1 + \sqrt5}{2} \;=\; \varphi, \qquad
y_{k+1} \;=\; x_k + \frac{1 - 1}{\varphi}(x_k - x_{k-1}) \;=\; x_k.
$$

The trajectory continues from $(t_{k+1}, y_{k+1}) = (\varphi,\ x_k)$, exactly as if the algorithm had been re-initialized at $x_k$.

*Incorrect but plausible.* Setting $x_{k-1} \to x_k$ while leaving $t_k$ unchanged produces $y_{k+1} = x_k$ as well (the extrapolation direction is zero), but leaves $t_{k+1} = (1 + \sqrt{1 + 4t_k^2})/2$ at the un-restarted value. The two implementations diverge in $t_{k+1}$ and nowhere else.

**Why the distinction matters.** $t_{k+1}$ feeds into the next extrapolation coefficient $(t_{k+1} - 1)/t_{k+2}$; if the restart leaves $t_{k+1}$ at its un-restarted value, the next few iterates get a larger-than-intended overshoot and the very acceleration the restart was supposed to damp remains in force. Over many restarts the two implementations produce different trajectories and converge at different rates. The correct override-then-update semantics is what the third subproblem function must implement.

## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math


def test_case_1():
    t_next, y_next = fista_momentum_update(1.0, np.array([0.0]), np.array([0.0]))
    golden = (1 + math.sqrt(5)) / 2
    assert abs(t_next - golden) < 1e-12
    print('test_case_1 passed')


def test_case_2():
    t_k = 2.3
    x_prev = np.array([1.0, -1.0, 0.5])
    x_curr = np.array([1.4, -0.6, 0.9])
    t_next, y_next = fista_momentum_update(t_k, x_prev, x_curr)
    expected_t = (1 + math.sqrt(1 + 4 * t_k ** 2)) / 2
    expected_y = x_curr + ((t_k - 1) / expected_t) * (x_curr - x_prev)
    assert abs(t_next - expected_t) < 1e-12
    assert np.allclose(y_next, expected_y, atol=1e-12)
    print('test_case_2 passed')


def test_case_3():
    x_prev = np.array([0.0, 0.0])
    x_curr = np.array([1.0, -1.0])
    for t_k in [1.0, 1.05, 1.5, 2.0, 5.0, 50.0, 1000.0]:
        t_next, _ = fista_momentum_update(t_k, x_prev, x_curr)
        assert t_next > t_k
    print('test_case_3 passed')


def test_case_4():
    x_same = np.array([2.0, -3.0, 0.5])
    for t_k in [1.0, 3.7, 42.0]:
        t_next, y_next = fista_momentum_update(t_k, x_same, x_same)
        assert np.allclose(y_next, x_same, atol=1e-12)
    print('test_case_4 passed')


def test_case_5():
    x_prev = np.array([0.0, 1.0])
    x_curr = np.array([1.0, 0.0])
    try:
        fista_momentum_update(0.5, x_prev, x_curr); assert False
    except ValueError: pass
    try:
        fista_momentum_update(float('nan'), x_prev, x_curr); assert False
    except ValueError: pass
    try:
        fista_momentum_update(1.0, x_prev, np.array([1.0, 0.0, 0.0])); assert False
    except ValueError: pass
    try:
        fista_momentum_update(1.0, np.array([1.0, np.inf]), x_curr); assert False
    except ValueError: pass
    print('test_case_5 passed')


def test_case_6():
    obj = np.array([5.0, 4.0, 3.5, 3.2, 3.1])
    assert fista_should_restart(obj) is False
    obj2 = np.array([5.0, 4.0, 4.5])
    assert fista_should_restart(obj2) is True
    obj3 = np.array([3.14])
    assert fista_should_restart(obj3) is False
    obj4 = np.array([2.0, 2.0, 2.0])
    assert fista_should_restart(obj4) is False
    print('test_case_6 passed')


def test_case_7():
    try:
        fista_should_restart(np.array([])); assert False
    except ValueError: pass
    try:
        fista_should_restart(np.array([1.0, np.nan])); assert False
    except ValueError: pass
    print('test_case_7 passed')


def test_case_8():
    x_prev = np.array([0.0, 0.0])
    x_curr = np.array([1.0, -1.0])
    obj_no_restart = np.array([5.0, 4.0, 3.0])
    for t_k in [1.0, 1.5, 3.0, 10.0]:
        t_next, y_next, restarted = fista_step_momentum(t_k, x_prev, x_curr, obj_no_restart)
        expected_t, expected_y = fista_momentum_update(t_k, x_prev, x_curr)
        assert restarted is False
        assert abs(t_next - expected_t) < 1e-12
        assert np.allclose(y_next, expected_y, atol=1e-12)
    print('test_case_8 passed')


def test_case_9():
    x_prev = np.array([1.0, -1.0])
    x_curr = np.array([1.5, -0.5])
    obj_restart = np.array([5.0, 6.0])
    golden = (1 + math.sqrt(5)) / 2
    for t_k in [1.0, 1.5, 2.3, 7.0, 100.0]:
        t_next, y_next, restarted = fista_step_momentum(t_k, x_prev, x_curr, obj_restart)
        assert restarted is True
        assert abs(t_next - golden) < 1e-12
        assert np.allclose(y_next, x_curr, atol=1e-15)
    print('test_case_9 passed')


def test_case_10():
    x_prev = np.array([1.0, -1.0])
    x_curr = np.array([1.5, -0.5])
    obj_restart = np.array([5.0, 6.0])
    ref_t, ref_y = fista_momentum_update(1.0, x_prev, x_curr)
    for t_k in [1.0, 3.0, 50.0]:
        t_next, y_next, restarted = fista_step_momentum(t_k, x_prev, x_curr, obj_restart)
        assert restarted is True
        assert abs(t_next - ref_t) < 1e-12
        assert np.allclose(y_next, ref_y, atol=1e-15)
    print('test_case_10 passed')


def test_case_11():
    x_prev = np.array([1.0, -1.0])
    x_curr = np.array([1.5, -0.5])
    obj_restart = np.array([5.0, 6.0])
    try:
        fista_step_momentum(0.5, x_prev, x_curr, obj_restart)
        assert False, 'expected ValueError for t_k < 1 (validation precedes restart)'
    except ValueError:
        pass
    print('test_case_11 passed')


def test_case_12():
    x_prev = np.array([1.0, -1.0])
    x_curr = np.array([1.5, -0.5])
    obj_restart = np.array([5.0, 6.0])
    try:
        fista_step_momentum(float('nan'), x_prev, x_curr, obj_restart)
        assert False, 'expected ValueError for NaN t_k (validation precedes restart)'
    except ValueError:
        pass
    print('test_case_12 passed')


def test_case_13():
    x_prev = np.array([0.0, 1.0])
    x_curr = np.array([1.0, 0.0])
    obj_no_restart = np.array([3.0, 2.0])
    try:
        fista_step_momentum(0.5, x_prev, x_curr, obj_no_restart)
        assert False, 'expected ValueError for t_k < 1'
    except ValueError:
        pass
    try:
        fista_step_momentum(1.0, x_prev, np.array([1.0, 0.0, 0.0]), obj_no_restart)
        assert False, 'expected ValueError for mismatched shapes'
    except ValueError:
        pass
    print('test_case_13 passed')

## Solution

In [ ]:
import numpy as np
import math


def fista_momentum_update(t_k, x_prev, x_curr):
    if not isinstance(t_k, (int, float, np.number)) or not np.isfinite(t_k) or t_k < 1:
        raise ValueError("t_k must be finite and >= 1")

    x_prev = np.asarray(x_prev, dtype=float)
    x_curr = np.asarray(x_curr, dtype=float)
    if x_prev.ndim != 1 or x_curr.ndim != 1 or x_prev.shape != x_curr.shape:
        raise ValueError("x_prev and x_curr must be 1-D arrays of the same shape")
    if not np.all(np.isfinite(x_prev)) or not np.all(np.isfinite(x_curr)):
        raise ValueError("x_prev and x_curr must contain only finite values")

    t_next = (1.0 + math.sqrt(1.0 + 4.0 * t_k ** 2)) / 2.0
    y_next = x_curr + ((t_k - 1.0) / t_next) * (x_curr - x_prev)
    return float(t_next), y_next


def fista_should_restart(obj_history):
    obj_history = np.asarray(obj_history, dtype=float)
    if obj_history.ndim != 1 or obj_history.size == 0:
        raise ValueError("obj_history must be a non-empty 1-D array")
    if not np.all(np.isfinite(obj_history)):
        raise ValueError("obj_history must contain only finite values")
    if obj_history.size < 2:
        return False
    return bool(obj_history[-1] > obj_history[-2])


def fista_step_momentum(t_k, x_prev, x_curr, obj_history):
    # Validation precedes the restart override.
    if not isinstance(t_k, (int, float, np.number)) or not np.isfinite(t_k) or t_k < 1:
        raise ValueError("t_k must be finite and >= 1")

    restarted = fista_should_restart(obj_history)
    t_effective = 1.0 if restarted else float(t_k)
    t_next, y_next = fista_momentum_update(t_effective, x_prev, x_curr)
    return t_next, y_next, restarted

# Main Problem

## Prompt

Combine the proximal-gradient step (Subproblem 1) and the momentum extrapolation (Subproblem 2) into the full **FISTA** solver for the LASSO problem

$$\min_{x\in\mathbb{R}^n} \; F(x) = \tfrac12\|Ax-b\|_2^2 + \lambda\|x\|_1.$$

Algorithm (Beck & Teboulle, 2009):

1. Compute $L = \|A\|_2^2$ (the squared spectral norm of $A$, i.e. the largest eigenvalue of $A^TA$).
2. Initialize $x_0 = 0$, $y_1 = x_0$, $t_1 = 1$.
3. For $k = 1, 2, \dots$: compute $\nabla f(y_k) = A^T(Ay_k - b)$, then
   $x_k = \texttt{prox\_gradient\_step}(y_k, \nabla f(y_k), L, \lambda)$ (Subproblem 1, applied at $y_k$, **not** at $x_{k-1}$).
4. Compute $(t_{k+1}, y_{k+1}) = \texttt{fista\_momentum\_update}(t_k, x_{k-1}, x_k)$ (Subproblem 2).
5. Stop when $\|x_k - x_{k-1}\|_\infty < \texttt{tol}$ or after `max_iter` iterations, whichever comes first.

```python
def fista_lasso_solver(A, b, lam, max_iter=1000, tol=1e-8):
    """
    Solve min_x 0.5*||Ax-b||_2^2 + lam*||x||_1 with FISTA.

    Parameters
    ----------
    A : np.ndarray, shape (m, n)
        Design matrix. Must be a finite, real 2-D array.
    b : np.ndarray, shape (m,)
        Target vector. Must be a finite 1-D array with length m, matching
        A's number of rows.
    lam : float
        L1 regularization strength. Must be non-negative and finite.
    max_iter : int, optional
        Maximum number of FISTA iterations. Must be a positive integer.
        Default 1000.
    tol : float, optional
        Convergence tolerance on the infinity norm of the update
        ||x_k - x_{k-1}||_inf. Must be positive and finite. Default 1e-8.

    Returns
    -------
    x : np.ndarray, shape (n,)
        The computed solution.
    n_iter : int
        The number of FISTA iterations actually performed. If A is the
        zero matrix (L = 0), the solver short-circuits and returns 0.
    obj_history : np.ndarray, shape (n_iter,)
        The objective value F(x_k) at the end of each iteration, in order.

    Raises
    ------
    ValueError
        If A is not a finite 2-D array; if b is not a finite 1-D array of
        length equal to A's row count; if lam is not non-negative and
        finite; if max_iter is not a positive integer; or if tol is not
        positive and finite.
    """
    ...
    return x, n_iter, obj_history

## Background

This main problem assembles the complete FISTA pipeline. Each piece already tested in isolation (the shrinkage step and the momentum update) must now be wired together correctly: in particular, the proximal-gradient step (Subproblem 1) must be evaluated at the *extrapolated* point $y_k$, while the momentum update (Subproblem 2) must be fed the two most recent *proximal-gradient outputs* $x_{k-1}, x_k$ -- not $y_{k-1}, y_k$. Swapping these is a common assembly mistake that would still run without error and would still converge, just at the slower, non-accelerated rate, which several tests below are designed to catch.

**A closed-form check when $A$ has orthonormal columns.** When $A^TA = I$ (so $L=1$), the LASSO problem decouples into $n$ independent scalar problems, each solved exactly by one soft-thresholding of $A^Tb$: $x^\star = S_\lambda(A^Tb)$. This gives an exact, closed-form answer to check the whole solver against, independent of any iterative-convergence argument (Test Case 1).

**The headline theoretical guarantee: $O(1/k^2)$, not merely "eventually converges."** Beck and Teboulle's central result is

$$
F(x_k) - F^\star \;\le\; \frac{2L\|x_0 - x^\star\|_2^2}{(k+1)^2}.
$$

Plain ISTA (proximal gradient without extrapolation, i.e. Subproblem 1 applied directly at $x_{k-1}$ every iteration) only guarantees the weaker $O(1/k)$ rate. On a well-conditioned problem both methods converge to machine precision quickly enough that the difference is not visible within a modest iteration budget; the gap becomes visible precisely when the problem is ill-conditioned enough that neither method has fully converged. Test Case 5 constructs such a problem and checks that the FISTA solver reaches a specific objective gap within a fixed iteration budget.

**An optimality certificate independent of any specific solver or closed form.** At a minimizer of a convex but non-smooth objective, Fermat's rule requires $0 \in \partial F(x^\star)$. For the LASSO objective this splits into: at every zero coordinate, $|[A^T(Ax-b)]_i| \le \lambda$ (the subgradient of $|x_i|$ at $0$ is the interval $[-1,1]$ scaled by $\lambda$); at every nonzero coordinate, $[A^T(Ax-b)]_i = -\lambda\,\mathrm{sign}(x_i)$ exactly. This holds at *any* correct solution to *any* valid LASSO instance, so it is a strong, general-purpose check that does not rely on knowing the answer in advance (Test Case 4).

**A known limit.** As $\lambda \to 0^+$ with $A$ of full column rank, the LASSO solution converges to the ordinary least-squares solution $x^\star = (A^TA)^{-1}A^Tb$ (Test Case 2).

**Degenerate design matrix.** If $A$ is the zero matrix, then $L = \|A\|_2^2 = 0$ and the objective reduces to $F(x) = \tfrac12\|b\|_2^2 + \lambda\|x\|_1$, which is minimized at $x = 0$ for any $\lambda > 0$. The solver must handle this case without dividing by $L$ or returning non-finite values: it should short-circuit and return $x = 0$ with zero iterations.

## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math


def test_case_1():
    # Orthonormal-columns closed form.
    H = np.array([[1, 1, 1, 1],
                  [1, -1, 1, -1],
                  [1, 1, -1, -1],
                  [1, -1, -1, 1]], dtype=float) / 2.0
    b = np.array([1.0, -2.0, 0.5, 3.0])
    lam = 0.6
    x, n_iter, obj_hist = fista_lasso_solver(H, b, lam, max_iter=5000, tol=1e-14)
    z = H.T @ b
    closed_form = np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)
    assert isinstance(n_iter, int)
    assert np.allclose(x, closed_form, atol=1e-5), f"expected {closed_form}, got {x}"
    print('test_case_1 passed')


def test_case_2():
    # lam -> 0 with full column rank A recovers OLS.
    A = np.array([
        [3.0, 1.0, 0.0],
        [1.0, 2.0, 1.0],
        [0.0, 1.0, 4.0],
        [2.0, 0.0, 1.0],
        [1.0, 1.0, 1.0],
    ])
    b = np.array([1.0, 2.0, 3.0, 0.5, 1.5])
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam=1e-9, max_iter=20000, tol=1e-16)
    x_ols = np.linalg.solve(A.T @ A, A.T @ b)
    assert np.allclose(x, x_ols, atol=1e-3), f"expected OLS {x_ols}, got {x}"
    print('test_case_2 passed')


def test_case_3():
    # Golden regression on a fixed, hand-checked problem.
    A = np.array([
        [ 2.0,  0.0,  1.0, -1.0],
        [ 0.0,  1.5, -0.5,  0.5],
        [ 1.0, -1.0,  2.0,  0.0],
        [-1.0,  0.5,  0.0,  1.0],
        [ 0.5,  0.0, -1.0,  2.0],
    ])
    b = np.array([1.0, -0.5, 2.0, 0.5, -1.5])
    lam = 0.4
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=20000, tol=1e-14)
    expected = np.array([-0.0952381, 0.0, 1.0619048, -0.0142857])
    assert np.allclose(x, expected, atol=1e-4), f"expected {expected}, got {x}"
    print('test_case_3 passed')


def test_case_4():
    # KKT / subgradient optimality certificate.
    A = np.array([
        [ 2.0,  0.0,  1.0, -1.0],
        [ 0.0,  1.5, -0.5,  0.5],
        [ 1.0, -1.0,  2.0,  0.0],
        [-1.0,  0.5,  0.0,  1.0],
        [ 0.5,  0.0, -1.0,  2.0],
    ])
    b = np.array([1.0, -0.5, 2.0, 0.5, -1.5])
    lam = 0.4
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=20000, tol=1e-14)
    grad = A.T @ (A @ x - b)
    zero_mask = np.abs(x) < 1e-6
    nonzero_mask = ~zero_mask
    assert np.all(np.abs(grad[zero_mask]) <= lam + 1e-3)
    assert np.allclose(grad[nonzero_mask], -lam * np.sign(x[nonzero_mask]), atol=1e-3)
    print('test_case_4 passed')


def test_case_5():
    # Acceleration: FISTA must reach a tight objective gap within 800 iters
    # on an ill-conditioned problem where plain ISTA cannot.
    m, n = 80, 50
    i_idx = np.arange(m)
    base = np.sin(0.37 * i_idx + 0.5)
    A = np.zeros((m, n))
    for j in range(n):
        A[:, j] = base + 0.0015 * (j + 1) * np.cos(0.19 * i_idx * (j + 3) + 0.1 * j)
    b = 2.0 * base + 0.1 * np.sin(0.7 * i_idx + 1.0)
    lam = 0.02
    F_star = 0.20015545983174493
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=800, tol=1e-15)
    F_fista = 0.5 * np.linalg.norm(A @ x - b)**2 + lam * np.sum(np.abs(x))
    assert F_fista - F_star < 1e-2, (
        f"FISTA should reach objective gap < 1e-2 within 800 iterations; "
        f"got gap={F_fista - F_star:.4e}"
    )
    print('test_case_5 passed')


def test_case_6():
    # Validation tier.
    A = np.array([[1.0, 0.0], [0.0, 1.0]])
    b = np.array([1.0, 1.0])

    for kwargs, label in [
        (dict(lam=-0.1), 'negative lam'),
        (dict(lam=0.1, max_iter=0), 'max_iter=0'),
        (dict(lam=0.1, max_iter=10.5), 'non-integer max_iter'),
        (dict(lam=0.1, tol=-1e-6), 'non-positive tol'),
    ]:
        try:
            fista_lasso_solver(A, b, **kwargs)
            assert False, f'expected ValueError for {label}'
        except ValueError:
            pass

    try:
        fista_lasso_solver(A, np.array([1.0, 1.0, 1.0]), lam=0.1)
        assert False, 'expected ValueError for mismatched A/b shapes'
    except ValueError:
        pass

    try:
        fista_lasso_solver(np.array([1.0, 2.0, 3.0]), b, lam=0.1)
        assert False, 'expected ValueError for non-2-D A'
    except ValueError:
        pass

    print('test_case_6 passed')


def test_case_7():
    # Full-trajectory replication check: an independent implementation
    # of the exact algorithm as stated in the prompt -- L = ||A||_2^2;
    # x_k = prox_gradient_step applied at the extrapolated point y_k
    # (not at x_{k-1}); (t_{k+1}, y_{k+1}) = fista_momentum_update fed
    # the two most recent proximal-gradient outputs x_{k-1}, x_k (not
    # y_{k-1}, y_k); stop on ||x_k - x_{k-1}||_inf < tol or max_iter --
    # must match the solver's x, n_iter, and obj_history exactly. This
    # catches the "common assembly mistake" the Background warns about
    # (e.g. evaluating the gradient step at x_{k-1} instead of y_k),
    # which would still run and still converge, just without full
    # verification against the prompt's specified wiring.
    def reference_fista(A, b, lam, max_iter, tol):
        m, n = A.shape
        L = np.linalg.norm(A, 2) ** 2
        if L == 0.0:
            return np.zeros(n), 0, np.zeros(0)
        x_prev = np.zeros(n)
        y = np.zeros(n)
        t_k = 1.0
        obj_hist = []
        x_curr = x_prev.copy()
        n_iter = 0
        for k in range(1, max_iter + 1):
            grad_y = A.T @ (A @ y - b)
            z = y - grad_y / L
            x_curr = np.sign(z) * np.maximum(np.abs(z) - lam / L, 0.0)
            obj = 0.5 * np.linalg.norm(A @ x_curr - b) ** 2 + lam * np.sum(np.abs(x_curr))
            obj_hist.append(obj)
            n_iter = k
            t_next = (1.0 + math.sqrt(1.0 + 4.0 * t_k ** 2)) / 2.0
            y = x_curr + ((t_k - 1.0) / t_next) * (x_curr - x_prev)
            t_k = t_next
            if np.max(np.abs(x_curr - x_prev)) < tol:
                x_prev = x_curr
                break
            x_prev = x_curr
        return x_curr, n_iter, np.array(obj_hist)

    A = np.array([
        [1.3, -0.7, 0.2],
        [0.4,  1.1, -0.9],
        [-0.6, 0.3, 1.5],
        [0.8,  0.5, 0.1],
    ])
    b = np.array([1.0, -0.5, 0.7, 0.2])
    lam = 0.25
    max_iter, tol = 500, 1e-12

    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=max_iter, tol=tol)
    x_ref, n_iter_ref, obj_hist_ref = reference_fista(A, b, lam, max_iter, tol)

    assert n_iter == n_iter_ref, f"expected n_iter={n_iter_ref}, got {n_iter}"
    assert np.allclose(x, x_ref, atol=1e-10), f"expected {x_ref}, got {x}"
    assert np.allclose(obj_hist, obj_hist_ref, atol=1e-10)
    print('test_case_7 passed')


def test_case_8():
    # Iteration count bookkeeping: n_iter must be an int, equal the
    # number of recorded objective values, reflect early stopping via
    # tol (not always run to max_iter on an easy, well-conditioned
    # problem), and stay stable once convergence has been reached even
    # if max_iter is increased further.
    A = np.array([[1.0, 0.0], [0.0, 1.0]])
    b = np.array([1.0, 1.0])
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam=0.1, max_iter=1000, tol=1e-10)
    assert isinstance(n_iter, int)
    assert n_iter == len(obj_hist)
    assert 0 < n_iter < 1000, f"expected early stopping, got n_iter={n_iter}"

    x2, n_iter2, obj_hist2 = fista_lasso_solver(A, b, lam=0.1, max_iter=5000, tol=1e-10)
    assert np.allclose(x, x2, atol=1e-10)
    assert n_iter2 == len(obj_hist2)
    print('test_case_8 passed')


def test_case_9():
    # Objective-history bookkeeping: shape (n_iter,), all finite, the
    # last entry equal to F evaluated at the returned x, and an overall
    # decrease from the first recorded iterate to the last (FISTA's
    # objective sequence need not be monotone step-to-step, but it must
    # still make net progress here).
    A = np.array([
        [2.0, 0.0, 1.0],
        [0.0, 1.5, -0.5],
        [1.0, -1.0, 2.0],
    ])
    b = np.array([1.0, -0.5, 2.0])
    lam = 0.3
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=2000, tol=1e-13)
    assert obj_hist.shape == (n_iter,)
    assert np.all(np.isfinite(obj_hist))
    F_final = 0.5 * np.linalg.norm(A @ x - b) ** 2 + lam * np.sum(np.abs(x))
    assert np.isclose(obj_hist[-1], F_final, atol=1e-9)
    assert obj_hist[-1] < obj_hist[0]
    print('test_case_9 passed')


def test_case_10():
    # Degenerate zero design matrix: L = ||A||_2^2 = 0, so the solver
    # must short-circuit rather than divide by zero, returning x = 0
    # with zero iterations and an empty objective history.
    A = np.zeros((4, 3))
    b = np.array([1.0, -2.0, 0.5, 3.0])
    lam = 0.7
    x, n_iter, obj_hist = fista_lasso_solver(A, b, lam, max_iter=100, tol=1e-8)
    assert isinstance(n_iter, int)
    assert n_iter == 0
    assert np.allclose(x, np.zeros(3))
    assert obj_hist.shape == (0,)
    print('test_case_10 passed')


## Solution

In [ ]:
import numpy as np
import math


def prox_l1_weighted(u, theta, weights=None):
    """Proximal operator of the weighted L1 norm."""
    u = np.asarray(u, dtype=float)
    if u.ndim != 1:
        raise ValueError("u must be a 1-D array")
    if not np.all(np.isfinite(u)):
        raise ValueError("u must contain only finite values")
    if not np.isfinite(theta) or theta < 0:
        raise ValueError(f"theta must be non-negative and finite, got {theta!r}")

    if weights is None:
        w = np.ones_like(u)
    else:
        w = np.asarray(weights, dtype=float)
        if w.ndim != 1 or w.shape != u.shape:
            raise ValueError("weights must be a 1-D array of the same shape as u")
        if not np.all(np.isfinite(w)):
            raise ValueError("weights must contain only finite values")
        if np.any(w < 0):
            raise ValueError("weights must be non-negative")

    return np.sign(u) * np.maximum(np.abs(u) - theta * w, 0.0)


def prox_gradient_step(x, grad, L, lam, weights=None):
    """One proximal-gradient (ISTA) step, with optional weighted L1."""
    x = np.asarray(x, dtype=float)
    grad = np.asarray(grad, dtype=float)

    if x.ndim != 1 or grad.ndim != 1 or x.shape != grad.shape:
        raise ValueError("x and grad must be 1-D arrays of the same shape")
    if not np.all(np.isfinite(x)):
        raise ValueError("x must contain only finite values")
    if not np.all(np.isfinite(grad)):
        raise ValueError("grad must contain only finite values")
    if not np.isfinite(L) or L <= 0:
        raise ValueError(f"L must be a positive finite number, got L={L!r}")
    if not np.isfinite(lam) or lam < 0:
        raise ValueError(f"lam must be a non-negative finite number, got lam={lam!r}")

    z = x - grad / L
    return prox_l1_weighted(z, lam / L, weights=weights)


def fista_momentum_update(t_k, x_prev, x_curr):
    if not isinstance(t_k, (int, float, np.number)) or not np.isfinite(t_k) or t_k < 1:
        raise ValueError("t_k must be finite and >= 1")

    x_prev = np.asarray(x_prev, dtype=float)
    x_curr = np.asarray(x_curr, dtype=float)
    if x_prev.ndim != 1 or x_curr.ndim != 1 or x_prev.shape != x_curr.shape:
        raise ValueError("x_prev and x_curr must be 1-D arrays of the same shape")
    if not np.all(np.isfinite(x_prev)) or not np.all(np.isfinite(x_curr)):
        raise ValueError("x_prev and x_curr must contain only finite values")

    t_next = (1.0 + math.sqrt(1.0 + 4.0 * t_k ** 2)) / 2.0
    y_next = x_curr + ((t_k - 1.0) / t_next) * (x_curr - x_prev)
    return float(t_next), y_next


def fista_should_restart(obj_history):
    obj_history = np.asarray(obj_history, dtype=float)
    if obj_history.ndim != 1 or obj_history.size == 0:
        raise ValueError("obj_history must be a non-empty 1-D array")
    if not np.all(np.isfinite(obj_history)):
        raise ValueError("obj_history must contain only finite values")
    if obj_history.size < 2:
        return False
    return bool(obj_history[-1] > obj_history[-2])


def fista_step_momentum(t_k, x_prev, x_curr, obj_history):
    # Validation precedes the restart override.
    if not isinstance(t_k, (int, float, np.number)) or not np.isfinite(t_k) or t_k < 1:
        raise ValueError("t_k must be finite and >= 1")

    restarted = fista_should_restart(obj_history)
    t_effective = 1.0 if restarted else float(t_k)
    t_next, y_next = fista_momentum_update(t_effective, x_prev, x_curr)
    return t_next, y_next, restarted


def fista_lasso_solver(A, b, lam, max_iter=1000, tol=1e-8):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    if A.ndim != 2:
        raise ValueError(f"A must be a 2-D array, got shape {A.shape}")
    if not np.all(np.isfinite(A)):
        raise ValueError("A must contain only finite values")
    m, n = A.shape
    if b.ndim != 1 or b.shape[0] != m:
        raise ValueError(f"b must be a 1-D array of length {m}, got shape {b.shape}")
    if not np.all(np.isfinite(b)):
        raise ValueError("b must contain only finite values")
    if not np.isfinite(lam) or lam < 0:
        raise ValueError(f"lam must be non-negative and finite, got lam={lam!r}")
    if isinstance(max_iter, bool) or not isinstance(max_iter, (int, np.integer)) or max_iter <= 0:
        raise ValueError(f"max_iter must be a positive integer, got {max_iter!r}")
    if not np.isfinite(tol) or tol <= 0:
        raise ValueError(f"tol must be positive and finite, got tol={tol!r}")

    L = np.linalg.norm(A, 2) ** 2

    # Zero-design-matrix case: L = 0, A = 0. The objective reduces to
    # lam * ||x||_1, minimized at x = 0. Return without iterating.
    if L == 0.0:
        return np.zeros(n), 0, np.zeros(0)

    x_prev = np.zeros(n)
    y = np.zeros(n)
    t_k = 1.0
    x_curr = x_prev.copy()
    obj_hist = []

    n_iter = 0
    for k in range(1, max_iter + 1):
        grad_y = A.T @ (A @ y - b)
        x_curr = prox_gradient_step(y, grad_y, L, lam)
        obj = 0.5 * np.linalg.norm(A @ x_curr - b) ** 2 + lam * np.sum(np.abs(x_curr))
        obj_hist.append(obj)
        n_iter = k

        t_next, y = fista_momentum_update(t_k, x_prev, x_curr)
        t_k = t_next

        if np.max(np.abs(x_curr - x_prev)) < tol:
            x_prev = x_curr
            break
        x_prev = x_curr

    return x_curr, n_iter, np.array(obj_hist)
